# Attention-Based Topics (ABT) 

##### This is a minimal version of the code (without figure plots). It is used to measure computational costs (running time and memory usage).

## 1. Install Packages

In [ ]:
!pip install -U sentence-transformers
!pip install -U gensim

## 2. Topic definition

In [2]:
import time
start_time = time.time()

import psutil
process = psutil.Process()

corpus = []
vocabulary = []    
topics = []
coherence = 0.0

class Topic:
    def __init__(self, index):
        self.words = []
        self.sentences = ""
        self.coherence = 0.0
        self.number_of_sentences = 0
        self.index = index

## 3. Preprocessing Input
This block fills the `corpus` variable with the contents of the input file.

### 3.1. Using MovieLens as input [Default]

In [3]:
import os.path

DATASET = 'movielens'
DATASET_PATH = 'input/'+DATASET+'/movies.csv'

BERT_MODEL = 'all-mpnet-base-v2'
# # BERT_MODEL = 'fagner/envoy'
# # BERT_MODEL = 'dmis-lab/biobert-base-cased-v1.1'
# # BERT_MODEL = 'facebook/bart-base'
# # BERT_MODEL = 'bert-base-cased'


OUTPUT_PATH = 'output/'
if not os.path.exists(OUTPUT_PATH): os.mkdir(OUTPUT_PATH)
OUTPUT_PATH += DATASET + '/'
if not os.path.exists(OUTPUT_PATH): os.mkdir(OUTPUT_PATH)
OUTPUT_PATH += BERT_MODEL + '/'
if not os.path.exists(OUTPUT_PATH): os.makedirs(OUTPUT_PATH, exist_ok=True)

import pandas as pd
import re

start_time_A = time.time()

df = pd.read_csv(DATASET_PATH) 
titles = df['title']

for i in range(len(titles)):
    s1 = re.split(r'\s+\((\d+)\)', titles[i])
    
    if df['genres'][i] != '(no genres listed)':
        corpus.append(s1[0] + '|' + df['genres'][i])
    else:
        corpus.append(s1[0])

### 3.2. Using CliCR as input [Alternative]

In [4]:
# import json
# import os.path


# DATASET = 'clicr'
# DATASET_PATH = 'input/'+DATASET+'/train1.0.json'

# BERT_MODEL = 'bert-base-cased'

# OUTPUT_PATH = 'output/'
# if not os.path.exists(OUTPUT_PATH): os.mkdir(OUTPUT_PATH)
# OUTPUT_PATH += DATASET + '/'
# if not os.path.exists(OUTPUT_PATH): os.mkdir(OUTPUT_PATH)
# OUTPUT_PATH += BERT_MODEL + '/'
# if not os.path.exists(OUTPUT_PATH): os.makedirs(OUTPUT_PATH, exist_ok=True)

# start_time_A = time.time()

# with open(DATASET_PATH) as f:
#     dataset = json.load(f)

#     for datum in dataset["data"]:
#         title = datum["document"]["title"].replace("BEG__", "").replace("__END", "")
#         if (title != 'This article has a correction'):
#             corpus.append(title)

## 4. Method Implementation
### Step 1: Sentence Modeling

In [5]:
from sentence_transformers import SentenceTransformer
    
language_model = SentenceTransformer(BERT_MODEL)
vector_space = language_model.encode(corpus)

print("Cost of step 1")
print("Running time: %s seconds" % (time.time() - start_time_A))
gb_rss = process.memory_info().rss / (1024 * 1024 * 1024)
print('Memory usage: %s GB' % gb_rss)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 1 cost
Running time: 188.06993627548218 seconds
Memory usage: 1.40643310546875 GB


### Step 2: Hierarchical Sentence Aggregation

In [6]:
DISTANCE_THRESHOLD = 300

In [7]:
from sklearn.cluster import AgglomerativeClustering

start_time_B = time.time()

clustering_model = AgglomerativeClustering(linkage='ward', distance_threshold=DISTANCE_THRESHOLD, n_clusters=None)
clustering_model = clustering_model.fit(vector_space)
k = clustering_model.n_clusters_

print('Distance Threshold: ', DISTANCE_THRESHOLD)
print('Resulting Clusters:',  k)

print("\nCost of step 2")
print("Running time: %s seconds" % (time.time() - start_time_B))
gb_rss = process.memory_info().rss / (1024 * 1024 * 1024)
print('Memory usage: %s GB' % gb_rss)

Distance Threshold:  300
Resulting Clusters: 1

Step 2 cost
Running time: 22.977924823760986 seconds
Memory usage: 1.4048500061035156 GB


### Step 3: Representing Topics

In [8]:
MAX_DF = 0.001
c_MAX_DF = 0.99999
TOP_WORDS = 10

In [9]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csc_matrix

start_time_C = time.time()

topics = [Topic(i) for i in range(k)]
for i in range(len(corpus)):
    cluster_index = clustering_model.labels_[i]
    topics[cluster_index].sentences += corpus[i] + " "
    topics[cluster_index].number_of_sentences += 1

if (k > 1):
    c_tfidf_model = TfidfVectorizer(max_df=c_MAX_DF)
    c_tfidf = c_tfidf_model.fit_transform([topic.sentences for topic in topics])
    c_tfidf_matrix = c_tfidf.toarray()
    vocabulary = c_tfidf_model.get_feature_names_out()
    
    for i, topic in enumerate(topics):
        sorted_term_indexes = np.argsort(-1*c_tfidf_matrix[topic.index])
        topic.words = [vocabulary[j] for j in sorted_term_indexes]   
else:
    c_tfidf_model = TfidfVectorizer(max_df=MAX_DF)    
    c_tfidf = c_tfidf_model.fit_transform(corpus)
    c_tfidf_matrix = c_tfidf.toarray()
    vocabulary = c_tfidf_model.get_feature_names_out()
    
    mean_tfidf = np.array(c_tfidf_matrix.mean(axis=0)).flatten()
    sorted_term_indexes = np.argsort(-1*mean_tfidf)
    topics[0].words = [vocabulary[j] for j in sorted_term_indexes]
            
print("Cost of step 3")

print("Running time: %s seconds" % (time.time() - start_time_C))
gb_rss = process.memory_info().rss / (1024 * 1024 * 1024)
print('Memory usage: %s GB' % gb_rss)

print("\nTotal running time: %s seconds" % (time.time() - start_time))

Step 3 cost
Running time: 0.46099853515625 seconds
Memory usage: 1.4621009826660156 GB

Total running time: 212.1088991165161 seconds


## 5. Validation

In [10]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

tfidf_model = TfidfVectorizer(max_df=MAX_DF)

tfidf = tfidf_model.fit_transform(corpus)
words_by_sentence = tfidf_model.inverse_transform(tfidf)

dictionary = Dictionary(words_by_sentence)

cm = CoherenceModel(topics=[topic.words for topic in topics], texts=words_by_sentence, dictionary=dictionary, coherence="c_v",topn=TOP_WORDS)

coherence = cm.get_coherence()
coherence_per_topic = cm.get_coherence_per_topic()

for i, c in enumerate(coherence_per_topic):
    topics[i].coherence = c
    print(topics[i].words[:TOP_WORDS])

print('Total coherence: ', coherence)
print('Coherence by topic: ', coherence_per_topic)

['apes', 'furious', 'heat', 'thin', 'pie', 'drive', 'kings', 'ride', 'dracula', 'panther']
Total coherence:  0.6084505225350373
Coherence by topic:  [0.6084505225350373]


In [11]:
with open(OUTPUT_PATH + '/results.txt', "a") as file:
    print('Hyper-parameters: \n{Language Model: ' + BERT_MODEL + ', Distance Threshold: '+str(DISTANCE_THRESHOLD) + ', TfIdf Threshold: '+str(MAX_DF) + ', cTfIdf Threshold: '+str(c_MAX_DF) + ', Top Words: '+str(TOP_WORDS) +'}', file=file)   
    print("Results: \n{Number of topics: "+str(len(topics)) + ', Vocabulary length: '+str(len(vocabulary)) + ', Total Coherence: '+str(coherence) + "}\n", file=file)  

    for topic in topics:
        print('Topic '+str(topic.index), file=file)    
        print('Number of sentences: '+str(topic.number_of_sentences), file=file)
        print('Coherence: '+str(topic.coherence), file=file)
        print('Top Words: '+str(topic.words[:TOP_WORDS]), file=file)  
        print('', file=file)

    print('----------------------------------------------------------------------------', file=file)  